In [ ]:
#| hide
from kernmini_magics.core import *

# kernmini_magics

> Shared machinery for IPython magics that embed a kernmini shell

Run a kernmini shell inside IPython as a `%%magic`: the same engine as the standalone kernel, from a Python notebook.

## Developer Guide

If you are new to using `nbdev` here are some useful pointers to get you started.

### Install kernmini_magics in Development mode

```sh
# make sure kernmini_magics package is installed in development mode
$ pip install -e .

# make changes under nbs/ directory
# ...

# compile to have changes apply to kernmini_magics
$ nbdev_prepare
```

## Usage

### Installation

Install latest from the GitHub [repository][repo]:

```sh
$ pip install git+https://github.com/Rahuketu86/kernmini_magics.git
```

or from [conda][conda]

```sh
$ conda install -c Rahuketu86 kernmini_magics
```

or from [pypi][pypi]


```sh
$ pip install kernmini_magics
```


[repo]: https://github.com/Rahuketu86/kernmini_magics
[docs]: https://Rahuketu86.github.io/kernmini_magics/
[pypi]: https://pypi.org/project/kernmini_magics/
[conda]: https://anaconda.org/Rahuketu86/kernmini_magics

### Documentation

Documentation can be found hosted on this GitHub [repository][repo]'s [pages][docs]. Additionally you can find package manager specific guidelines on [conda][conda] and [pypi][pypi] respectively.

[repo]: https://github.com/Rahuketu86/kernmini_magics
[docs]: https://Rahuketu86.github.io/kernmini_magics/
[pypi]: https://pypi.org/project/kernmini_magics/
[conda]: https://anaconda.org/Rahuketu86/kernmini_magics

## How to use

A [kernmini](https://github.com/AnswerDotAI/kernmini) shell -- the object with `execute()` / `complete()` / `inspect()` / `set_stream_sender()` / `interrupt()` that kernmini serves as a Jupyter kernel -- can also run inside an ordinary IPython kernel. `Workspace` wraps one as a `%name` / `%%name` magic plus a Python object, so a Python notebook can mix in another language and call back and forth. Each kernmini kernel repo defines its own magic on top of this:

| Package | Magics | Object | Quiet marker | `register()` |
|---|---|---|---|---|
| `prolog_wasm_kernmini` | `%%prolog` / `%prolog` | `prolog` | trailing `;` | yes, called from Prolog with `py_call/2` |
| `chibi_wasm_kernmini` | `%%scheme` / `%scheme` | `scheme` | trailing `;` | yes, `snake_case` becomes `kebab-case` |
| `quickjs-kernmini` | `%%js`, `%%ts` (one shared context) | `js` (= `ts`) | trailing `;;` | yes, sync or async |
| `micropython-wasi-kernmini` | `%%mpy` / `%mpy` | `mpy` | trailing `;` | yes |
| `clang-repl-kernmini` | `%%c` / `%c` | `c` | none (C's own rule) | no |
| `web_kernel` | `%%tab` / `%tab` | `tab` | trailing `;;` | yes, sync or async; page code calls it as `await fn(...)` |

Each is loaded explicitly, e.g. `%load_ext prolog_wasm_kernmini.magics`, and installed with that package's `magics` extra. Each repo's `nbs/11_demo_python_magic.ipynb` is a Python dialog using its magic.

### What every magic does

- **`%%name` cell / `%name` line.** The cell runs in the kernel's own shell, so state persists across cells exactly as in the standalone kernel. The line form, and calling the object (`prolog("...")`), return a string: the implicit result's text, or for Prolog the answer text.
- **Errors** raise a per-language `MagicError` subclass (`PrologError`, `SchemeError`, `JSError`, `MicroPythonError`, `ClangError`) after any output produced before the error; `e.output` holds that output. IPython shows the kernel's message, not Python frames.
- **A trailing quiet marker** hides the implicit result, as in basedpl's and IPython's own magics; output the code prints itself still shows.
- **`obj.register(fn)`** (also a decorator) makes a Python function callable from the other language. It does in a Python kernel what a `%%host` cell does in the standalone kernel; `%%host` itself is unchanged there.
- **Ctrl-C** stops a runaway cell through the shell's `interrupt()`. **`obj.timeout`** (seconds, off by default; env `<NAME>_MAGIC_TIMEOUT` or `KERNMINI_MAGIC_TIMEOUT`) does the same on a deadline and raises `TimeoutError`. The workspace stays usable after both.
- **Completion** inside a magic cell uses the shell's `complete()`; `obj.names(prefix)` returns the same names.
- **`%name --reset`** / `obj.reset()` starts a fresh shell.

### Defining a magic for a kernel

```python
from kernmini_magics import Workspace, MagicError

class FooError(MagicError): "A failed Foo cell."

def _make_shell():
    from .core import FooKernminiShell      # the kernel's own shell class, unchanged
    return FooKernminiShell()

async def _register(shell, name, fn):         # optional: expose a Python callable to Foo
    await shell._ensure_context()
    shell._rt.register(name, fn)

foo = Workspace('foo', _make_shell, error_cls=FooError, register=_register,
                quiet_marker=';', strip_marker=True)

def load_ipython_extension(ip): foo.load(ip)
```

`Workspace` arguments:

- `quiet_marker` / `strip_marker` -- the trailing marker that hides the result (`None` for none), and whether to remove it before running (for languages where it would be meaningless or invalid).
- `call_returns` -- `'result'` (default: return the implicit result's text, print output) or `'output'` (return the printed text itself, not echoed).
- `aliases` -- more magic names for the same engine.
- `timeout` -- default deadline in seconds; otherwise read from the environment.

A shell that accepts `quiet=` in `execute()` is told when the marker was used, so it can hide results it prints as output (Prolog's bindings). A shell with `interrupt()` gets real Ctrl-C; without it, the running task is cancelled.

### How it works

- Every shell call runs on one private daemon thread with its own event loop, so a synchronous magic can drive async shells without `asyncio.run()` inside ipykernel's running loop, and a thread-bound engine always sees the same thread.
- Output events cross to the main thread through a queue and are printed there, so they are attributed to the running cell.
- While a magic waits, it installs `signal.default_int_handler` for SIGINT: for magic cells ipykernel installs a handler that cancels the cell's asyncio future, which never takes effect while a blocking magic holds the thread.
- The completion matcher works out the text before the cursor from IPython's line + column coordinates and filters the shell's reply to the typed fragment (names starting with it, else names containing it), since some shells rely on the frontend to filter.